In [1]:
import os
import pandas as pd
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

C:\Users\ADMIN\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CSV_FILE = "/kaggle/input/newdatasetnlp/data_1.csv"
TEXT_COL = "Thông tin"
LABEL_COL = "Label"
STT_COL = "STT"
THRESH = 0.80
SLEEP = 0.5      
LABEL_SUFFIX = ""

df = pd.read_csv("/kaggle/input/newdatasetnlp/data_1.csv")
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL] != ""].copy()

In [3]:
embedder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
def similarity_vi_en(vi: str, en: str) -> float:
    e_vi = embedder.encode(vi, normalize_embeddings=True)
    e_en = embedder.encode(en, normalize_embeddings=True)
    return float(cos_sim(e_vi, e_en))

In [5]:
!pip install deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.1 MB/s eta 0:00:00


In [2]:
# Translation
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source="vi", target="en")

In [3]:
CSV_FILE = "data.csv"
with open(CSV_FILE, encoding="utf-8-sig") as f:
    next_stt = sum(1 for _ in f)

In [9]:
import time
OUTPUT_CSV = "/kaggle/working/data_translation.csv"
rows = []
stt = 1

for idx, row in df.iterrows():
    vi = row[TEXT_COL]
    label = row[LABEL_COL]

    try:
        en = translator.translate(vi).strip()
    except Exception as e:
        print(f"[{idx}] translate error:", e)
        continue

    if not en:
        continue

    score = similarity_vi_en(vi, en)
    keep = score >= THRESH
    print(f"[{idx}] score={score:.3f} keep={keep}")

    if score >= THRESH:
        rows.append({
            STT_COL: stt,
            TEXT_COL: en,
            LABEL_COL: label + LABEL_SUFFIX
        })
        stt += 1

    time.sleep(SLEEP)

out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("✅ Saved to:", OUTPUT_CSV)

[0] score=0.790 keep=False
[1] score=0.646 keep=False
[2] score=0.951 keep=True
[3] score=0.920 keep=True
[4] score=0.974 keep=True
[5] score=0.924 keep=True
[6] score=0.895 keep=True
[7] score=0.963 keep=True
[8] score=0.973 keep=True
[9] score=0.889 keep=True
[10] score=0.849 keep=True
[11] score=0.908 keep=True
[12] score=0.960 keep=True
[13] score=0.927 keep=True
[14] score=0.832 keep=True
[15] score=0.729 keep=False
[16] score=0.961 keep=True
[17] score=0.973 keep=True
[18] score=0.923 keep=True
[19] score=0.937 keep=True
[20] score=0.745 keep=False
[21] score=0.774 keep=False
[22] score=0.878 keep=True
[23] score=0.935 keep=True
[24] score=0.890 keep=True
[25] score=0.889 keep=True
[26] score=0.927 keep=True
[27] score=0.926 keep=True
[28] score=0.889 keep=True
[29] score=0.900 keep=True
[30] score=0.813 keep=True
[31] score=0.866 keep=True
[32] score=0.870 keep=True
[33] score=0.970 keep=True
[34] score=0.843 keep=True
[35] score=0.682 keep=False
[36] score=0.867 keep=True
[37] 

OSError: Cannot save file into a non-existent directory: '/kaggle/working/newdatasetnlp'

In [10]:
OUTPUT_CSV = "/kaggle/working/data_translation.csv"
out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Success")

Success


In [11]:
df_out = pd.read_csv(OUTPUT_CSV)

In [12]:
df_out

,STT,Thông tin,Label
0,1,ICITE 2025 held for the first time in Vietnam:...,Nghiên cứu
1,2,AI in ABB's smart power system: A step forward...,Nghiên cứu
2,3,ACM RACS 2025 International Conference at IUH:...,Nghiên cứu
3,4,IUH Young Science Conference 2025: Gathering y...,Nghiên cứu
4,5,"Workshop ""Management and commercialization of ...",Nghiên cứu
...,...,...,...
522,523,The function of the Department of Political Af...,Giới thiệu
523,524,The function of the Department of Political Af...,Giới thiệu
524,525,The function of the Department of Political Af...,Giới thiệu
525,526,The function of the Department of Political Af...,Giới thiệu
